In [1]:
# This file is a jupytext-paired Python script export of
# `comparing_campaigns_solution.ipynb`. The canonical artifact for learners is
# the notebook (.ipynb); this script is provided for code review and `git diff`
# readability. Run `jupytext --sync` to keep the two in lockstep after edits.

# Comparing Growth Campaigns with Expected Utility (SOLUTION)

## Scenario

You are the lead decision scientist at **Fernbrook Stays**, a small
short-term-rental management firm that lists properties on Airbnb-style
platforms. The VP of Growth wants to launch a campaign next quarter to
attract new property owners onto the Fernbrook Stays platform, and has put three
options on the table:

- **Featured Placement Push** — invest in professional photography, top-of-search
  placement, and paid listing on aggregator sites. Expensive,
  with high upside if the broader short-term-rental market is strong but a
  sizeable loss if the market softens.
- **Neighborhood Concierge Bundle** — bundle Fernbrook Stays listings with a local
  concierge service (airport pickup, restaurant booking, late-night
  support). Modest budget, modest upside, modest downside.
- **Hold** — defer the campaign one quarter and reinvest the budget into
  existing-host retention.

Fernbrook Stays operates across a set of mid-tier short-term-rental markets. The
campaign is a *single firm-wide choice* — whichever option the team picks
runs across Fernbrook Stays's whole portfolio next quarter; we are not choosing
different campaigns for different cities. How well that single choice
performs depends on **the demand cycle the short-term-rental industry is
in over the next three months** — a Strong-demand, Average, or Weak-demand
environment that affects all of Fernbrook Stays's markets at once. The cycle is
uncertain (we decide the campaign now; the cycle reveals itself later),
but it is not a coin flip — there is a real distribution we can estimate
from data.

To estimate that distribution, we use cross-city variation in current
Inside Airbnb data as a *stand-in* for the range of conditions any given
market can be in at any given time. The reasoning: eighteen real cities
span a wide range of revenue-proxy values today, and that cross-sectional
spread is a defensible empirical picture of what a Strong, Average, or
Weak rental environment looks like.

The file `data/airbnb_city_stats.csv` contains city-level summary
statistics for 18 well-known short-term-rental markets in the format
published by Inside Airbnb (see `data/README.md` for the full citation).
For each city, we have the median nightly price, the median monthly
reviews per listing (Inside Airbnb's standard occupancy proxy), and a few
other fields. We compute `median_price × median_reviews_per_month` — a
per-listing monthly revenue proxy — for each city, tertile-bin the
eighteen cities into Strong / Average / Weak environments, and use the
empirical share of cities in each bin as the prior on what kind of
environment Fernbrook Stays will face next quarter. The cities themselves are
not Fernbrook Stays's markets; they are a benchmark for what "a Strong market"
or "a Weak market" looks like in numbers.

## What this notebook delivers

The VP of Growth has asked the team to run the numbers — expected value,
expected utility under risk aversion, and minimax regret — and to identify
which decision rule the team would lean on if forced to commit today. A
full stakeholder-facing recommendation is a separate skill covered in other
course, after additional analysis steps. Today's deliverable is:

1. The side-by-side comparison of what each decision rule selects.
2. A 1–2 sentence defended choice of decision rule.

## Setup

In [2]:
import numpy as np
import pandas as pd

DATA_PATH = "../comparing-marketing-campaigns-starter/data/airbnb_city_stats.csv"

## 1. Load the data

In [3]:
cities = pd.read_csv(DATA_PATH)
print(cities.head())
print("...")
print(cities.tail())

            city        country  total_listings  median_price_usd  \
0  New York City  United States           37000               200   
1    Los Angeles  United States           30000               240   
2         Austin  United States           13000               240   
3      Vancouver         Canada            5500               170   
4    Mexico City         Mexico           25000                60   

   median_reviews_per_month  pct_entire_home  
0                      0.70               52  
1                      0.50               68  
2                      0.55               72  
3                      0.55               55  
4                      0.85               80  
...
      city    country  total_listings  median_price_usd  \
13  Madrid      Spain           22000               110   
14    Rome      Italy           28000               125   
15  Lisbon   Portugal           25000               100   
16   Tokyo      Japan           12000                95   
17  

## 2. Per-listing monthly revenue proxy

A coarse but defensible per-listing monthly revenue indicator: median nightly
price times Inside Airbnb's standard occupancy proxy (median monthly reviews
per active listing).

In [4]:
cities["revenue_proxy"] = (
    cities["median_price_usd"] * cities["median_reviews_per_month"]
)
print(cities[["city", "median_price_usd", "median_reviews_per_month",
              "revenue_proxy"]].sort_values("revenue_proxy").round(2))

              city  median_price_usd  median_reviews_per_month  revenue_proxy
5     Buenos Aires                55                      0.65          35.75
16           Tokyo                95                      0.40          38.00
10          Berlin                80                      0.55          44.00
6   Rio de Janeiro                70                      0.70          49.00
4      Mexico City                60                      0.85          51.00
9           London               120                      0.50          60.00
8            Paris               110                      0.60          66.00
7        Cape Town               100                      0.70          70.00
13          Madrid               110                      0.75          82.50
3        Vancouver               170                      0.55          93.50
17          Sydney               195                      0.50          97.50
15          Lisbon               100                      1.00  

## 3. Classify each city into Strong / Average / Weak by tertile

In [5]:
t33, t67 = cities["revenue_proxy"].quantile([1 / 3, 2 / 3])
print(f"33rd percentile cutoff: {t33:.2f}")
print(f"67th percentile cutoff: {t67:.2f}")


def classify(value: float) -> str:
    if value >= t67:
        return "Strong"
    if value >= t33:
        return "Average"
    return "Weak"


cities["environment"] = cities["revenue_proxy"].apply(classify)
print(cities[["city", "revenue_proxy", "environment"]])

33rd percentile cutoff: 64.00
67th percentile cutoff: 102.08
              city  revenue_proxy environment
0    New York City         140.00      Strong
1      Los Angeles         120.00      Strong
2           Austin         132.00      Strong
3        Vancouver          93.50     Average
4      Mexico City          51.00        Weak
5     Buenos Aires          35.75        Weak
6   Rio de Janeiro          49.00        Weak
7        Cape Town          70.00     Average
8            Paris          66.00     Average
9           London          60.00        Weak
10          Berlin          44.00        Weak
11       Amsterdam         110.00      Strong
12       Barcelona         117.00      Strong
13          Madrid          82.50     Average
14            Rome         106.25      Strong
15          Lisbon         100.00     Average
16           Tokyo          38.00        Weak
17          Sydney          97.50     Average


## 4. Empirical probability of each environment

In [6]:
counts = cities["environment"].value_counts()
p = (counts / counts.sum()).reindex(["Strong", "Average", "Weak"])
print("Empirical probabilities:")
print(p.round(3))
print(f"Sum: {p.sum():.3f}")

Empirical probabilities:
environment
Strong     0.333
Average    0.333
Weak       0.333
Name: count, dtype: float64
Sum: 1.000


By construction, the tertile bins each contain ~1/3 of the cities. This
gives a simple, defensible baseline distribution to work from.

## 5. Payoff matrix

In [7]:
payoffs = pd.DataFrame(
    {
        "Strong":  {"Featured Placement Push": 9.0, "Neighborhood Concierge Bundle": 2.7, "Hold": 0.0},
        "Average": {"Featured Placement Push": 2.2, "Neighborhood Concierge Bundle": 1.4, "Hold": 0.0},
        "Weak":    {"Featured Placement Push": -5.5, "Neighborhood Concierge Bundle": 0.2, "Hold": 0.0},
    }
)
print("Payoff matrix ($M incremental profit):")
print(payoffs)

Payoff matrix ($M incremental profit):
                               Strong  Average  Weak
Featured Placement Push           9.0      2.2  -5.5
Neighborhood Concierge Bundle     2.7      1.4   0.2
Hold                              0.0      0.0   0.0


## 6. Expected value per option

In [8]:
ev = (payoffs * p).sum(axis=1)
print("EV per option ($M):")
print(ev.round(3))
print(f"\nEV-maximizing option: {ev.idxmax()}  (${ev.max():.2f}M)")

EV per option ($M):
Featured Placement Push          1.900
Neighborhood Concierge Bundle    1.433
Hold                             0.000
dtype: float64

EV-maximizing option: Featured Placement Push  ($1.90M)


## 7. Expected CRRA utility per option

CRRA utility on (wealth_baseline + incremental profit), with γ = 2 and
W = $50M. The wealth baseline keeps utility well-defined and represents
the existing scale of the business; with γ > 1 the function is more sensitive
to losses than to equally sized gains, which is what "risk aversion" means
operationally here.

In [9]:
GAMMA = 2.0
WEALTH_M = 50.0


def crra_utility(profit_m, gamma: float = GAMMA, wealth_m: float = WEALTH_M):
    """CRRA utility evaluated at (wealth_m + profit_m). Vectorized."""
    x = wealth_m + np.asarray(profit_m, dtype=float)
    if abs(gamma - 1.0) < 1e-9:
        return np.log(x)
    return (np.power(x, 1.0 - gamma) - 1.0) / (1.0 - gamma)


utility_table = payoffs.map(crra_utility)
expected_utility = (utility_table * p).sum(axis=1)
ce_total_wealth = np.power(
    expected_utility * (1.0 - GAMMA) + 1.0, 1.0 / (1.0 - GAMMA)
)
certainty_equivalent = ce_total_wealth - WEALTH_M

print("Expected utility per option:")
print(expected_utility.round(6))
print("\nCertainty-equivalent profit ($M) per option:")
print(certainty_equivalent.round(3))
print(f"\nUtility-maximizing option: {expected_utility.idxmax()}  "
      f"(CE = ${certainty_equivalent[expected_utility.idxmax()]:.2f}M)")

Expected utility per option:
Featured Placement Push          0.980474
Neighborhood Concierge Bundle    0.980550
Hold                             0.980000
dtype: float64

Certainty-equivalent profit ($M) per option:
Featured Placement Push          1.214
Neighborhood Concierge Bundle    1.413
Hold                            -0.000
dtype: float64

Utility-maximizing option: Neighborhood Concierge Bundle  (CE = $1.41M)


Notice that *Featured Placement Push* has the highest **expected profit** but
*Neighborhood Concierge Bundle* has the highest **certainty-equivalent profit**.
A risk-averse decision-maker would trade away expected profit to
avoid the -$5.5M downside in a Weak environment.

## 8. Minimax regret per option

In [10]:
best_per_state = payoffs.max(axis=0)
regret = best_per_state - payoffs
max_regret = regret.max(axis=1)

print("Regret matrix ($M):")
print(regret)
print("\nMax regret per option ($M):")
print(max_regret)
print(f"\nMinimax-regret option: {max_regret.idxmin()}  "
      f"(max regret = ${max_regret.min():.2f}M)")

Regret matrix ($M):
                               Strong  Average  Weak
Featured Placement Push           0.0      0.0   5.7
Neighborhood Concierge Bundle     6.3      0.8   0.0
Hold                              9.0      2.2   0.2

Max regret per option ($M):
Featured Placement Push          5.7
Neighborhood Concierge Bundle    6.3
Hold                             9.0
dtype: float64

Minimax-regret option: Featured Placement Push  (max regret = $5.70M)


## 9. Compare the three decision rules

In [11]:
comparison = pd.DataFrame(
    {
        "Selected option": [ev.idxmax(),
                            expected_utility.idxmax(),
                            max_regret.idxmin()],
        "Score": [f"EV = ${ev.max():.2f}M",
                  f"CE = ${certainty_equivalent[expected_utility.idxmax()]:.2f}M",
                  f"Max regret = ${max_regret.min():.2f}M"],
    },
    index=["EV-max", "Expected-utility-max (γ=2)", "Minimax regret"],
)
print(comparison)

                                          Selected option                Score
EV-max                            Featured Placement Push          EV = $1.90M
Expected-utility-max (γ=2)  Neighborhood Concierge Bundle          CE = $1.41M
Minimax regret                    Featured Placement Push  Max regret = $5.70M


**The three rules do not fully agree.** EV-max picks Featured Placement Push,
minimax-regret also picks Featured Placement Push, but expected CRRA utility (γ = 2)
picks Neighborhood Concierge Bundle. The disagreement is real: Featured Placement Push has
the higher expected profit but a heavier tail in a Weak market — its
certainty-equivalent profit is lower than Neighborhood Concierge Bundle's
once the -$5.5M downside is priced in.

**The decision rule I would lean on is expected CRRA utility.** With γ = 2,
the gap between Featured Placement Push's EV and Neighborhood Concierge Bundle's EV does
not justify the -$5.5M downside in a Weak quarter for a firm of Fernbrook Stays's
scale. A risk-neutral decision-maker should lean on EV-max instead.

This deliverable stops at *defending a decision rule*. Translating the
comparison into a stakeholder-facing recommendation — with caveats,
robustness checks, and downstream pipeline output — is built explicitly in
the communication-focused modules.

## 10. Segmentation-aware option: Targeted Placement Push

Every option so far commits Fernbrook Stays to the *same campaign across its entire
portfolio*. But Fernbrook Stays can observe which markets are currently Strong, Average,
or Weak using the Inside Airbnb data — and could target its spend accordingly.

**Targeted Placement Push**: run Featured Placement Push only in currently-Strong markets,
Neighborhood Concierge Bundle in Average markets, and Hold in Weak markets.

In [12]:
payoffs.loc["Targeted Placement Push"] = {"Strong": 5.5, "Average": 1.4, "Weak": -0.9}
print("Updated payoff matrix ($M):")
print(payoffs)

Updated payoff matrix ($M):
                               Strong  Average  Weak
Featured Placement Push           9.0      2.2  -5.5
Neighborhood Concierge Bundle     2.7      1.4   0.2
Hold                              0.0      0.0   0.0
Targeted Placement Push           5.5      1.4  -0.9


In [13]:
ev4           = (payoffs * p).sum(axis=1)
utility4      = payoffs.map(crra_utility)
exp_u4        = (utility4 * p).sum(axis=1)
ce4_wealth    = np.power(exp_u4 * (1.0 - GAMMA) + 1.0, 1.0 / (1.0 - GAMMA))
ce4           = ce4_wealth - WEALTH_M
best4         = payoffs.max(axis=0)
max_regret4   = (best4 - payoffs).max(axis=1)

comparison4 = pd.DataFrame(
    {
        "Selected option": [ev4.idxmax(),
                            exp_u4.idxmax(),
                            max_regret4.idxmin()],
        "Score": [f"EV = ${ev4.max():.2f}M",
                  f"CE = ${ce4[exp_u4.idxmax()]:.2f}M",
                  f"Max regret = ${max_regret4.min():.2f}M"],
    },
    index=["EV-max", "Expected-utility-max (γ=2)", "Minimax regret"],
)
print(comparison4)

                                    Selected option                Score
EV-max                      Targeted Placement Push          EV = $2.00M
Expected-utility-max (γ=2)  Targeted Placement Push          CE = $1.87M
Minimax regret              Targeted Placement Push  Max regret = $3.50M


**All three rules now agree on Targeted Placement Push.** The uniform options
force Fernbrook Stays to either accept the full -$5.5M downside
(Featured Placement Push) or cap the upside (Neighborhood Concierge Bundle). Targeted Placement Push
sidesteps that tradeoff: by concentrating the expensive campaign where it is most
likely to pay off and limiting exposure elsewhere, it captures most of
Featured Placement Push's upside while cutting its worst-case loss substantially.

This mirrors a broader "segmentation beats uniform strategy" idea: targeting
spend using an observable signal can dominate a one-size-fits-all choice on
all three decision metrics at once.